# RVC Whisper Encoder 결과 확인 노트북

새 Colab 세션에서 실행하면 Drive에 저장된 RVC 재학습 결과를 다시 읽어서 확인합니다.

- 최신 `results/rvc_whisper_encoder_*` 폴더 자동 탐색
- `best_model_rvc_whisper_encoder_lcnn.pt` checkpoint 정보 확인
- `eval_rvc_val.txt`, `eval_rvc_holdout_combined.txt`, holdout 개별 평가 txt 수치 파싱
- 보고서에 붙일 수 있는 요약 문장 생성


In [ ]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. 경로 설정
from pathlib import Path
import re
import json
import textwrap
import pandas as pd
import torch

DEEPVOICE_DIR = Path('/content/drive/MyDrive/deepvoice')
RESULTS_ROOT = DEEPVOICE_DIR / 'results'
MODEL_PATH = DEEPVOICE_DIR / 'best_model_rvc_whisper_encoder_lcnn.pt'

# 특정 결과 폴더를 직접 지정하려면 문자열로 입력하세요.
# 예: RESULT_DIR_OVERRIDE = '/content/drive/MyDrive/deepvoice/results/rvc_whisper_encoder_20260628_085732'
RESULT_DIR_OVERRIDE = None

print('DEEPVOICE_DIR =', DEEPVOICE_DIR)
print('RESULTS_ROOT  =', RESULTS_ROOT)
print('MODEL_PATH    =', MODEL_PATH)


In [ ]:
# 3. 최신 RVC 결과 폴더 찾기
if RESULT_DIR_OVERRIDE:
    RESULT_DIR = Path(RESULT_DIR_OVERRIDE)
else:
    candidates = [p for p in RESULTS_ROOT.glob('rvc_whisper_encoder_*') if p.is_dir()]
    if not candidates:
        raise FileNotFoundError('results/rvc_whisper_encoder_* 결과 폴더를 찾지 못했습니다.')
    RESULT_DIR = max(candidates, key=lambda p: p.stat().st_mtime)

print('RESULT_DIR =', RESULT_DIR)
print('exists     =', RESULT_DIR.exists())

txt_files = sorted(RESULT_DIR.rglob('*.txt'))
json_files = sorted(RESULT_DIR.rglob('*.json'))

print('\n=== txt files ===')
for p in txt_files:
    print(p)

print('\n=== json files ===')
for p in json_files:
    print(p)


In [ ]:
# 4. 새 RVC 모델 checkpoint 정보 확인
print('model exists:', MODEL_PATH.exists())
if MODEL_PATH.exists():
    print('model size MB:', MODEL_PATH.stat().st_size / 1024 / 1024)
    ckpt = torch.load(MODEL_PATH, map_location='cpu')
    if isinstance(ckpt, dict):
        print('checkpoint keys:', list(ckpt.keys()))
        print('model_type:', ckpt.get('model_type'))
        print('val_f1:', ckpt.get('val_f1'))
        print('threshold:', ckpt.get('threshold'))
        cfg = ckpt.get('config', {}) or {}
        print('\n=== config 핵심 ===')
        for key in [
            'data_dir', 'output_path', 'real_train_dirs', 'real_val_dirs',
            'fake_train_dirs', 'fake_val_dirs', 'epochs', 'batch_size',
            'whisper_size', 'freeze_whisper'
        ]:
            print(f'{key}:', cfg.get(key))
    else:
        print('state_dict only checkpoint')
else:
    print('모델 파일이 없습니다. 학습이 끝났는지 확인하세요.')


In [ ]:
# 5. 평가 txt 파싱 함수
def read_text(path):
    return Path(path).read_text(encoding='utf-8', errors='replace')

def parse_eval_text(text):
    data = {}

    m = re.search(r'model:\s*(.+)', text)
    if m:
        data['model'] = m.group(1).strip()

    m = re.search(r'threshold:\s*([0-9.]+)', text)
    if m:
        data['threshold'] = float(m.group(1))

    # folder counts: "name: 123" lines
    counts = {}
    for name, count in re.findall(r'^([A-Za-z0-9_./-]+):\s*(\d+)(?:\s|$)', text, flags=re.MULTILINE):
        if name not in {'threshold', 'model'}:
            counts[name] = int(count)
    data['counts'] = counts

    # classification report rows
    for label in ['real', 'fake']:
        m = re.search(rf'^\s*{label}\s+([0-9.]+)\s+([0-9.]+)\s+([0-9.]+)\s+(\d+)', text, flags=re.MULTILINE)
        if m:
            data[f'{label}_precision'] = float(m.group(1))
            data[f'{label}_recall'] = float(m.group(2))
            data[f'{label}_f1_report'] = float(m.group(3))
            data[f'{label}_support'] = int(m.group(4))

    m = re.search(r'^\s*accuracy\s+([0-9.]+)\s+(\d+)', text, flags=re.MULTILINE)
    if m:
        data['accuracy'] = float(m.group(1))
        data['support_total'] = int(m.group(2))

    m = re.search(r'^F1:\s*([0-9.]+)', text, flags=re.MULTILINE)
    if m:
        data['f1'] = float(m.group(1))

    m = re.search(r'EER:\s*([0-9.]+)%\s*\(threshold=([0-9.eE+-]+),\s*FPR=([0-9.eE+-]+),\s*FNR=([0-9.eE+-]+)\)', text)
    if m:
        data['eer_percent'] = float(m.group(1))
        data['eer_threshold'] = float(m.group(2))
        data['eer_fpr'] = float(m.group(3))
        data['eer_fnr'] = float(m.group(4))

    m = re.search(r'min-DCF:\s*([0-9.eE+-]+)\s*/\s*normalized\s*([0-9.eE+-]+)\s*\(threshold=([0-9.eE+-]+)', text)
    if m:
        data['min_dcf'] = float(m.group(1))
        data['norm_min_dcf'] = float(m.group(2))
        data['min_dcf_threshold'] = float(m.group(3))

    m = re.search(r'best threshold scan:\s*([0-9.]+),\s*F1:\s*([0-9.]+)', text)
    if m:
        data['best_threshold_scan'] = float(m.group(1))
        data['best_threshold_f1'] = float(m.group(2))

    cm = re.search(r'confusion matrix.*?\n(\[\[.*?\]\])', text, flags=re.DOTALL)
    if cm:
        data['confusion_matrix'] = ' '.join(cm.group(1).split())

    return data

def parse_train_text(text):
    data = {}
    m = re.search(r'training complete \| best f1:\s*([0-9.]+)', text)
    if m:
        data['train_best_f1'] = float(m.group(1))
    epoch_matches = re.findall(r'Epoch\s+(\d+)/(\d+).*?best_f1:\s*([0-9.]+)\s*@\s*([0-9.]+).*?EER:\s*([0-9.]+)%.*?min-DCF:\s*([0-9.]+)', text)
    if epoch_matches:
        ep, total, f1, thr, eer, mindcf = epoch_matches[-1]
        data.update({
            'last_epoch': int(ep),
            'total_epochs': int(total),
            'last_best_f1': float(f1),
            'last_best_threshold': float(thr),
            'last_eer_percent': float(eer),
            'last_norm_min_dcf': float(mindcf),
        })
    return data


In [ ]:
# 6. 평가 결과 요약표 만들기
rows = []
for p in txt_files:
    rel = str(p.relative_to(RESULT_DIR))
    text = read_text(p)
    if p.name.startswith('eval_'):
        row = {'file': rel, 'kind': 'eval'}
        row.update(parse_eval_text(text))
        rows.append(row)
    elif p.name.startswith('train_'):
        row = {'file': rel, 'kind': 'train'}
        row.update(parse_train_text(text))
        rows.append(row)

df = pd.DataFrame(rows)
display_cols = [
    'file', 'kind', 'threshold', 'f1', 'fake_precision', 'fake_recall',
    'fake_f1_report', 'real_support', 'fake_support', 'accuracy',
    'eer_percent', 'norm_min_dcf', 'best_threshold_scan', 'best_threshold_f1',
    'confusion_matrix', 'train_best_f1', 'last_epoch', 'last_best_f1'
]
display_cols = [c for c in display_cols if c in df.columns]

print('RESULT_DIR:', RESULT_DIR)
print('rows:', len(df))
display(df[display_cols] if display_cols else df)

summary_csv = RESULT_DIR / 'rvc_metric_summary.csv'
df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
print('CSV 저장:', summary_csv)


In [ ]:
# 7. 결과 원문 전체 출력
for p in txt_files:
    print('\n' + '=' * 120)
    print(p)
    print('=' * 120)
    print(read_text(p))


In [ ]:
# 8. 보고서용 문장 자동 생성
eval_df = df[df.get('kind', '') == 'eval'].copy() if 'kind' in df.columns else pd.DataFrame()

def pick_row_contains(keyword):
    if eval_df.empty:
        return None
    hit = eval_df[eval_df['file'].str.contains(keyword, case=False, regex=False)]
    if len(hit) == 0:
        return None
    return hit.iloc[0]

combined = pick_row_contains('holdout_combined')
val = pick_row_contains('val')

sentences = []
if val is not None and pd.notna(val.get('f1')):
    sentences.append(
        f"RVC validation 평가에서는 F1 {val.get('f1'):.4f}, "
        f"EER {val.get('eer_percent'):.2f}%, normalized min-DCF {val.get('norm_min_dcf'):.4f}를 기록하였다."
    )

if combined is not None and pd.notna(combined.get('f1')):
    fake_support = int(combined.get('fake_support')) if pd.notna(combined.get('fake_support')) else None
    real_support = int(combined.get('real_support')) if pd.notna(combined.get('real_support')) else None
    support_part = ''
    if real_support is not None and fake_support is not None:
        support_part = f"real {real_support}개와 RVC holdout {fake_support}개 기준 "
    sentences.append(
        f"학습에 사용하지 않은 Joonjong 및 NELL_KLM43x4 RVC holdout에 대해 {support_part}"
        f"F1 {combined.get('f1'):.4f}, EER {combined.get('eer_percent'):.2f}%, "
        f"normalized min-DCF {combined.get('norm_min_dcf'):.4f}를 기록하였다."
    )

sentences.append(
    "다만 RVC 결과는 자체 구축 RVC 변환 데이터 기준의 추가 실험이므로, 최종 핵심 성능 주장은 TTS 탐지 모델 중심으로 유지하고 공인 데이터셋 기반 검증은 후속 과제로 정리하였다."
)

report_text = '\n'.join(sentences)
print(report_text)

out = RESULT_DIR / 'rvc_report_sentences.txt'
out.write_text(report_text, encoding='utf-8')
print('\n저장:', out)


## 해석할 때 주의

- `eval_rvc_val.txt`: KANE/Nell_V2에서 분리한 validation 결과입니다. 학습 RVC 모델과 같은 계열이라 높게 나올 수 있습니다.
- `eval_rvc_holdout_combined.txt`: 학습에 넣지 않은 Joonjong + NELL_KLM43x4 결과입니다. 보고서에 넣는다면 이 수치가 더 중요합니다.
- EER에 표시되는 threshold는 운영 threshold가 아니라 FPR/FNR이 만나는 분석용 threshold입니다.
- RVC 결과가 좋아도 TTS 최종 모델과 같은 강도로 일반화 성능을 주장하지 않는 편이 안전합니다.
